# pyLSTemp - Exemplo de `single_window(...)`

Este notebook demonstra como calcular LST pelo fluxo single-channel.

O fluxo recomendado e:

1. Calcular `brightness(...)` para a banda termal 10.
2. Chamar `single_window(...)` usando `brightness_band_10`, Red e NIR.

A funcao `single_window(...)` calcula o NDVI e a emissividade internamente.

## 1. Instalacao

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importacoes

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import brightness, single_window, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir metodos single-window

In [ ]:
catalog = list_algorithms()
print(list(catalog["single_channel"].keys()))

## 4. Ler bandas de entrada

Este exemplo usa:

- `B10`: banda termal 10.
- `B4`: Red.
- `B5`: NIR.

In [ ]:
data_dir = Path("data")

band_10_path = data_dir / "LC82210712016107LGN01_B10.tif"
red_path = data_dir / "LC82210712016107LGN01_B4.tif"
nir_path = data_dir / "LC82210712016107LGN01_B5.tif"

with rasterio.open(band_10_path) as src:
    band_10 = src.read(1).astype(float)
    profile = src.profile.copy()

with rasterio.open(red_path) as src:
    red = src.read(1).astype(float)

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype(float)

print("Band 10 shape:", band_10.shape)
print("Red shape:", red.shape)
print("NIR shape:", nir.shape)

## 5. Calcular brightness da banda 10

`single_window(...)` nao recebe DN termal diretamente. Primeiro calcule a temperatura de brilho.

In [ ]:
mask_10 = band_10 == 0

brightness_10 = brightness(
    band_10,
    band="band_10",
    sensor="landsat_8",
    mask=mask_10,
)

print("Brightness Band 10 mean:", np.nanmean(brightness_10))

## 6. Calcular LST single-window

O metodo padrao atual e `mono-window-2016`, usando emissividade `avdan-2016`.

In [ ]:
lst_single_celsius = single_window(
    brightness_band_10=brightness_10,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="mono-window-2016",
    emissivity_method="avdan-2016",
    unit="celsius",
)

print("LST single-window min:", np.nanmin(lst_single_celsius))
print("LST single-window max:", np.nanmax(lst_single_celsius))
print("LST single-window mean:", np.nanmean(lst_single_celsius))

## 7. Visualizar LST

In [ ]:
valid = lst_single_celsius[np.isfinite(lst_single_celsius)]
vmin, vmax = np.nanpercentile(valid, [2, 98])

plt.figure(figsize=(8, 6))
plt.imshow(lst_single_celsius, cmap="inferno", vmin=vmin, vmax=vmax)
plt.colorbar(label="LST (Celsius)")
plt.title("Single-window LST")
plt.axis("off")
plt.show()

## 8. Salvar resultado opcionalmente

In [ ]:
# output_path = Path("lst_single_window_celsius.tif")
# output_profile = profile.copy()
# output_profile.update(dtype="float32", nodata=np.nan, count=1)
#
# with rasterio.open(output_path, "w", **output_profile) as dst:
#     dst.write(lst_single_celsius.astype("float32"), 1)
#
# print("Arquivo salvo em:", output_path)